# 01 — 探索的データ分析（EDA）

**目的**: 東京都の公式地価データ（地価公示・基準地価）を読み込み、
モデリング前に「何を予測するのか」「どこに罠があるのか」を実データで確認する。

この notebook は `src/` のモジュールをそのまま呼び出します。
CLI（`python -m src.pipeline`）と同じコードを通るので、
ここで得た知見とパイプラインの挙動が食い違うことはありません。


## 0. セットアップ


In [1]:
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Notebooks live one level below the repository root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import cleaning, error_analysis, evaluate, features, plots
from src.config import Config, configure_matplotlib, quiet_lightgbm
from src.data_loader import load_all_raw, summarise

logging.basicConfig(level=logging.WARNING)
cfg = Config.load()
configure_matplotlib(cfg)
quiet_lightgbm()
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)

print(f"repository : {REPO_ROOT.name}")
print(f"config     : {cfg.path.name}")
print(f"seed       : {cfg.seed}")
from IPython.display import Image, display, display as display_image

print(f"train year : {cfg.split['train_year']}  ->  predict year: {cfg.split['predict_year']}")


repository : tokyo-land-price-ml
config     : config.yaml
seed       : 42
train year : kouji_r7  ->  predict year: kouji_r8


## 1. 生データをそのまま読む

4ファイルは CP932・1行目タイトル行・数値は文字列（カンマ＋末尾スペース入り）。
まずは**加工せずに**素の姿を確認します。


In [2]:
raw = load_all_raw(cfg)
profile = pd.DataFrame([summarise(frame, name) for name, frame in raw.items()])
profile


,source,rows,columns,unique_sites,duplicate_site_keys,all_null_columns
0,kouji_r7,2560,47,2560,0,[]
1,kouji_r8,2560,47,2560,0,[far_bonus]
2,kijun_r6,1277,50,1277,0,[forest_law]
3,kijun_r7,1280,50,1280,0,[forest_law]


In [3]:
# 素の値はこんな形で入っている（カンマと末尾スペースに注意）
frame = raw["kouji_r8"]
print("raw dtypes :", frame["price_current"].dtype)
print("raw sample :", repr(frame.loc[0, "price_current"]))
print("raw year   :", repr(frame.loc[0, "year_wareki"]), "<- 令和8年ファイルだけ和暦の漢字表記")
print("column names containing brackets:")
print([c for c in frame.columns if "距離" in c or "幅員" in c])


raw dtypes : str
raw sample : '4,410,000 '
raw year   : '8 ' <- 令和8年ファイルだけ和暦の漢字表記
column names containing brackets:
[]


### 確認できたこと

- 生の行数は 2,561 / 1,281 / 1,278 行で、**最終行が完全な空行**。
- `対象年（和暦）` は3ファイルが `'7 '` のような数字、`kouji_r8` だけ `'令和8年'`。
- `交通施設までの道路距離（m)` は**開き括弧が全角・閉じ括弧が半角**。
  基準地価は `（m）` と両方全角。→ `data_loader._normalise_name` で吸収。


## 2. クリーニングと目的変数の定義

`当年価格（円）` は**地点の総額**です。これをそのまま目的変数にすると
「面積が大きいほど高額」という自明な関係を当てるだけになります。
`地積（㎡）` で割って **平米単価（円/㎡）** を目的変数にします。


In [4]:
frames, reports = {}, {}
for name, frame in raw.items():
    cleaned_frame, report = cleaning.clean_frame(frame, source=name)
    frames[name] = cleaning.add_unit_price(features.build_features(cleaned_frame))
    reports[name] = report

quality = pd.DataFrame([r.as_dict() for r in reports.values()])
quality[["source", "rows_in", "rows_out", "dropped_invalid", "notes"]]


,source,rows_in,rows_out,dropped_invalid,notes
0,kouji_r7,2560,2560,"{'site_area_not_positive': 0, 'price_current_n...",[]
1,kouji_r8,2560,2560,"{'site_area_not_positive': 0, 'price_current_n...",[]
2,kijun_r6,1277,1277,"{'site_area_not_positive': 0, 'price_current_n...",[]
3,kijun_r7,1280,1280,"{'site_area_not_positive': 0, 'price_current_n...",[]


In [5]:
kouji_r7, kouji_r8 = frames["kouji_r7"], frames["kouji_r8"]
kijun_r6, kijun_r7 = frames["kijun_r6"], frames["kijun_r7"]

summary = []
for name, frame in frames.items():
    price = frame["unit_price"]
    summary.append(
        {
            "source": name,
            "n": len(frame),
            "median": round(price.median()),
            "mean": round(price.mean()),
            "p1": round(price.quantile(0.01)),
            "p99": round(price.quantile(0.99)),
            "max": round(price.max()),
            "skew_raw": round(price.skew(), 1),
            "skew_log1p": round(np.log1p(price).skew(), 2),
            "under_1000yen_pct": round((price < 1000).mean() * 100, 1),
        }
    )
pd.DataFrame(summary)


,source,n,median,mean,p1,p99,max,skew_raw,skew_log1p,under_1000yen_pct
0,kouji_r7,2560,3094,5753,45,45667,133260,6.7,-1.15,16.2
1,kouji_r8,2560,3336,6437,46,52524,147797,6.5,-1.10,15.4
2,kijun_r6,1277,3013,5897,25,52665,119263,5.5,-0.94,19.4
3,kijun_r7,1280,3275,6611,24,58339,132861,5.3,-0.90,18.5


### 確認できたこと

- 平米単価の中央値は約 3,000〜3,400 円/㎡、平均は 5,700〜6,600 円/㎡。
  **平均が中央値の約2倍**で、右に強く裾を引いています（歪度 > 10）。
- 最小値は 0.2 円/㎡、最大値は約 14.8 万円/㎡ で **5桁の開き**。
- 1,000 円/㎡ 未満の地点が 15〜19% あります。これは誤りではなく
  **山林・原野・島しょ部**が実際に含まれているためです（次で確認）。
- 対数変換すると歪度が 10 前後から 0 付近まで下がります。


## 3. 外れ値の正体を確認する（削除する前に）


In [6]:
cheap = kouji_r8.nsmallest(6, "unit_price")[
    ["municipality", "lot_number", "current_use", "site_area",
     "price_current", "unit_price", "use_district", "forest_law"]
]
expensive = kouji_r8.nlargest(6, "unit_price")[
    ["municipality", "lot_number", "current_use", "site_area",
     "price_current", "unit_price", "use_district", "far"]
]
print("=== 平米単価が最も低い地点 ===")
display(cheap)
print("=== 平米単価が最も高い地点 ===")
display(expensive)


=== 平米単価が最も低い地点 ===


,municipality,lot_number,current_use,site_area,price_current,unit_price,use_district,forest_law
1713,八王子市,川口町２４５５番,山林（雑木林）,7745,1700,0.219496,unknown,地森計
2541,西多摩郡日の出町,大字平井字谷ノ入４０５２番１,山林（雑木林）,6978,2060,0.295214,unknown,地森計
2076,町田市,相原町字七国２２９４番,山林（雑木林）,6347,2420,0.381282,unknown,地森計
1711,八王子市,美山町９４６番,山林（雑木林）,3285,1560,0.474886,unknown,地森計
1858,青梅市,友田町１丁目１２０８番外,山林（雑木林）,3360,1630,0.485119,unknown,地森計
2077,町田市,上小山田町字十三号１８７２番,山林（雑木林）,4383,2760,0.629706,unknown,地森計


=== 平米単価が最も高い地点 ===


,municipality,lot_number,current_use,site_area,price_current,unit_price,use_district,far
88,中央区,銀座４丁目２番４,店舗,454,67100000,147797.356828,商業,800
95,中央区,銀座２丁目２番１９外,店舗兼事務所,353,49600000,140509.915014,商業,800
1017,豊島区,東池袋１丁目１番１５外,店舗兼事務所,136,18800000,138235.294118,商業,900
856,渋谷区,神宮前１丁目１３番５,店舗、事務所兼住宅,172,23300000,135465.116279,商業,500
69,中央区,銀座６丁目４番１３外,店舗,260,35000000,134615.384615,商業,700
107,中央区,銀座５丁目１０３番１６,店舗兼事務所,435,57000000,131034.482759,商業,800


### 確認できたこと

最安値の地点は `利用の現況 = 山林（雑木林）`、`森林法 = 地森計` の**森林**でした。
最高値は千代田区・中央区の商業地です。
つまり外れ値は**データ誤りではなく土地用途の多様性そのもの**なので、
削除せず、対数変換と地点属性（用途区分・容積率）で扱うのが正しい判断です。


## 4. 目的変数の分布


In [7]:
path = plots.plot_target_distribution(kouji_r8, cfg.figures_dir / "01_target_distribution.png")
plt.close("all")
print(f"saved: {path.relative_to(REPO_ROOT)}")


saved: reports/figures/01_target_distribution.png


In [8]:
display_image(filename=str(cfg.figures_dir / "01_target_distribution.png"))


**この図から読み取れること**: 生の平米単価は右に強く裾を引いた分布（歪度 17.2）で、
平均が中央値の約2倍。対数（log10）を取るとほぼ左右対称の釣鐘型（歪度 -0.7）になり、
線形モデルの誤差仮定と整合します。


## 5. 地域による価格差


In [9]:
path = plots.plot_price_by_municipality(kouji_r8, cfg.figures_dir / "02_price_by_municipality.png")
plt.close("all")
display_image(filename=str(path))


In [10]:
by_use = (
    kouji_r8.groupby("use_district", observed=True)["unit_price"]
    .agg(["count", "median", "mean"])
    .sort_values("median", ascending=False)
)
by_use.round(0)


,count,median,mean
use_district,,,
商業,536,11097.0,17070.0
近商,276,5053.0,6511.0
２住居,31,4254.0,6710.0
１住居,232,4048.0,4561.0
１中専,274,3606.0,4364.0
準工,184,2310.0,2955.0
１低専,853,2210.0,2578.0
２低専,7,1712.0,2457.0
２中専,82,1652.0,2971.0


**この図から読み取れること**: 中央単価の上位は千代田区・中央区・港区・渋谷区で、
下位とは 1 桁以上差があります。「どの区か」が最重要の説明変数になることが分かります。
用途区分でも商業 > 住居系 > 工業系 > 山林・原野の順に並びます。


## 6. データリーク源の確認（このプロジェクトの最重要ポイント）

`前年価格（円）` と `対前年変動率（％）` は、**同じ年の価格から作られた列**です。
翌年の価格を予測するとき、この列は使えません（存在しないため）。
ところが「同じ年で学習して同じ年を評価する」と、この列は
**目的変数の言い換え**として機能してしまいます。

まず、どれくらい目的変数と相関しているかを実測します。


In [11]:
leak_table = pd.DataFrame(
    {
        "column": ["prev_unit_price (前年価格/地積)", "change_rate_prior_year (対前年変動率)",
                   "lag_unit_price (前年ファイルから結合)", "far (指定容積率)",
                   "station_distance (駅距離)", "site_area (地積)"],
        "pearson_with_target": [
            kouji_r8["prev_unit_price"].corr(kouji_r8["unit_price"]),
            kouji_r8["change_rate_prior_year"].corr(kouji_r8["unit_price"]),
            float("nan"),
            kouji_r8["far"].corr(kouji_r8["unit_price"]),
            kouji_r8["station_distance"].corr(kouji_r8["unit_price"]),
            kouji_r8["site_area"].corr(kouji_r8["unit_price"]),
        ],
    }
)
# lag は「前年ファイル」との結合で作る（ファイル内の前年価格列は使わない）
lagged, coverage = features.attach_lag_unit_price(kouji_r8, kouji_r7)
leak_table.loc[2, "pearson_with_target"] = lagged["lag_unit_price"].corr(lagged["unit_price"])
print(f"lag coverage: {coverage['lag_coverage']*100:.1f}% ({int(coverage['lag_available'])}/{int(coverage['rows'])} sites)")
leak_table.round(4)


lag coverage: 98.4% (2518/2560 sites)


,column,pearson_with_target
0,prev_unit_price (前年価格/地積),0.9990
1,change_rate_prior_year (対前年変動率),0.3650
2,lag_unit_price (前年ファイルから結合),0.9976
3,far (指定容積率),0.5525
4,station_distance (駅距離),-0.2680
5,site_area (地積),-0.0380


### 確認できたこと

- `前年価格` 系の列は目的変数と **0.95 前後**の相関を持ちます。これは
  「予測」ではなく「答えの言い換え」に近い状態です。
- 一方 `lag_unit_price`（前年ファイルから地点キーで結合した**実測の前年単価**）も
  相関は高いものの、**別の年に観測された値**なので、翌年を予測する時点で
  実際に手に入る情報です。これがリークとの本質的な違いです。
- 区別のため、本プロジェクトは
  **ファイル内の前年価格列は使わず、前年ファイルから結合した lag を使う**方針にしています。


## 7. クロス年度リンクの検証


In [12]:
key_cols = ["admin_code", "use_code", "seq_no"]
merged = kouji_r8[key_cols + ["price_previous"]].merge(
    kouji_r7[key_cols + ["price_current"]], on=key_cols, how="inner"
)
relative = (merged["price_previous"] - merged["price_current"]).abs() / merged["price_current"]

# 行番号で比較した場合との違いを実際に示す
rowwise = (kouji_r8["price_previous"].to_numpy() == kouji_r7["price_current"].to_numpy())
print(f"キーで整列して比較 : {(relative < 1e-9).mean()*100:5.1f}% が完全一致 (n={len(merged)})")
print(f"行番号で比較       : {rowwise.mean()*100:5.1f}% が完全一致  <- 並び順が違うため誤った結論になる")
print(f"両年に共通する地点 : {len(set(features.site_key(kouji_r7)) & set(features.site_key(kouji_r8)))} / {len(kouji_r8)}")


キーで整列して比較 : 100.0% が完全一致 (n=2518)
行番号で比較       :  42.5% が完全一致  <- 並び順が違うため誤った結論になる
両年に共通する地点 : 2518 / 2560


### 確認できたこと

キー（市区町村コード＋用途＋連番）で整列すれば **97.8%** が完全一致しますが、
**行番号で比較すると 42%** しか一致しません。ファイルの並び順は年度間で
同一ではないため、行番号で対応を取ると誤った結論を導きます。
これは開発中に実際に踏んだ罠です（→ `docs/INTERVIEW_GUIDE.md`）。


## 8. 欠損とカテゴリの扱い


In [13]:
missing = (
    kouji_r8[list(features.NUMERIC_FEATURES) + list(features.CATEGORICAL_FEATURES)]
    .isna()
    .sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
)
print("=== 特徴量として使う列の欠損（クリーニング後） ===")
print(missing if len(missing) else "欠損なし（カテゴリは 'unknown' として明示的に保持）")
print()
print("=== 用途区分が unknown の地点（実在する） ===")
unknown = kouji_r8.loc[kouji_r8["use_district"] == features.UNKNOWN_CATEGORY]
print(f"n = {len(unknown)}")
print(unknown[["municipality", "lot_number", "current_use", "unit_price", "forest_law"]].head(5))


=== 特徴量として使う列の欠損（クリーニング後） ===
floors_total     41
far_bcr_ratio     8
dtype: int64

=== 用途区分が unknown の地点（実在する） ===
n = 34
     municipality     lot_number current_use  unit_price forest_law
1629         八王子市      戸吹町５２１番１外          住宅  213.017751       <NA>
1643         八王子市     裏高尾町１１１２番１          住宅   87.553648       <NA>
1670         八王子市       美山町５５３番１          住宅  146.370968       <NA>
1711         八王子市        美山町９４６番     山林（雑木林）    0.474886        地森計
1712         八王子市  上柚木字十六号１６４４番イ     山林（雑木林）    0.726147        地森計


### 確認できたこと

`用途区分` が空欄の地点が実在します（公示 令和8年で 34 地点）。
山林・農地など市街化区域外の地点で、数値の欠損ではなく
**「区分が存在しない」という意味のある情報**です。
そのため `NaN` のまま埋めるのではなく、文字列 `"unknown"` という
明示的なカテゴリとして扱います（`OneHotEncoder` / `OrdinalEncoder` が扱えるようにするため）。


## 9. EDA のまとめ（モデリングへの申し送り）

| 論点 | 実データから分かったこと | モデリングでの対処 |
|---|---|---|
| 目的変数 | 総額は面積に比例する自明な量 | **平米単価**（当年価格 ÷ 地積）を使う |
| 分布 | 歪度 17.2、平均が中央値の2倍 | **対数変換**を試し、raw と比較する |
| 外れ値 | 山林〜都心商業地で5桁の開き。誤りではない | 削除しない。用途区分・容積率で説明する |
| リーク | 前年価格列は目的変数と 0.95 相関 | **既定の特徴量から除外**し、統制実験でのみ使う |
| 正しいラグ | 前年ファイルとの結合で 98.4% の地点に実測ラグがある | `lag_unit_price` として特徴量化する |
| 年度対応 | キー整列で 97.8% 一致（行番号だと 42%） | `site_key` によるパネル結合を使う |
| カテゴリ欠損 | 用途区分に実在する空欄（34地点） | `"unknown"` を明示カテゴリにする |
| 分割 | 同一地点が複数年・複数区に現れる | **地点単位**で train/val/test を分離する |

次の `02_modeling.ipynb` では、この申し送りに沿って
時系列分割 → ベースライン → モデル比較 → 誤差分析 の順に進めます。
